In [3]:
import yfinance as yf
import pandas as pd

esg = pd.read_csv("SP 500 ESG Risk Ratings.csv")
tickers = esg["Symbol"].dropna().unique().tolist()

# Pull YTD return per ticker
returns = []
for t in tickers:
    try:
        # Explicitly set auto_adjust=True to handle the FutureWarning
        data = yf.download(t, period="ytd", progress=False, auto_adjust=True)
        if len(data) > 1:
            ytd_return = (data["Close"].iloc[-1] / data["Close"].iloc[0] - 1) * 100
            # Access the float value using .iloc[0] to address the FutureWarning
            returns.append({"Symbol": t, "YTD_Return_Pct": round(float(ytd_return.iloc[0]), 2)})
    except Exception:
        continue

returns_df = pd.DataFrame(returns)
returns_df.to_csv("ytd_returns.csv", index=False)
print(returns_df.shape)

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['DAY']: YFPricesMissingError('possibly delisted; no price data found  (period=ytd) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['WRK']: YFPricesMissingError('possibly delisted; no price data found  (period=ytd) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['WBA']: YFPricesMissingError('possibly delisted; no price data found  (period=ytd) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['IPG']: YFPricesMissingError('possibly delisted; no price data found  (period=ytd) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['MMC']: YFPricesMissingError('possibly delisted; no price data found  (period=ytd) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR

(485, 2)


In [5]:
import sqlite3

esg = pd.read_csv("SP 500 ESG Risk Ratings.csv")
returns = pd.read_csv("ytd_returns.csv")

conn = sqlite3.connect(":memory:")
esg.to_sql("esg", conn, index=False)
returns.to_sql("returns", conn, index=False)

485

In [6]:
q1 = """
SELECT
  e.Symbol,
  e.Sector,
  e."Total ESG Risk score" AS ESG_Risk_Score,
  e."Controversy Level" AS Controversy_Level,
  e."ESG Risk Level" AS ESG_Risk_Level,
  r.YTD_Return_Pct
FROM esg e
JOIN returns r ON e.Symbol = r.Symbol
WHERE e."Total ESG Risk score" IS NOT NULL
  AND r.YTD_Return_Pct IS NOT NULL;
"""
joined = pd.read_sql(q1, conn)
print(joined.shape)
joined.to_sql("joined", conn, index=False)
joined.head()

(416, 6)


,Symbol,Sector,ESG_Risk_Score,Controversy_Level,ESG_Risk_Level,YTD_Return_Pct
0,EMN,Basic Materials,25.3,Moderate Controversy Level,Medium,17.86
1,DPZ,Consumer Cyclical,29.2,Moderate Controversy Level,Medium,-18.71
2,DVA,Healthcare,22.6,Moderate Controversy Level,Medium,51.79
3,DRI,Consumer Cyclical,27.5,Moderate Controversy Level,Medium,21.14
4,ZTS,Healthcare,18.8,Moderate Controversy Level,Low,-37.30


In [7]:
q2 = """
SELECT *,
  NTILE(5) OVER (ORDER BY ESG_Risk_Score) AS ESG_Risk_Quintile
FROM joined;
"""
quintiles = pd.read_sql(q2, conn)
quintiles.to_csv("esg_returns_joined.csv", index=False)
quintiles.groupby("ESG_Risk_Quintile")["YTD_Return_Pct"].mean().round(2)

,YTD_Return_Pct
ESG_Risk_Quintile,
1,16.86
2,12.50
3,12.93
4,9.03
5,16.07


In [8]:
from scipy import stats
import numpy as np

# Correlation: ESG risk score vs YTD return
corr, p_value = stats.pearsonr(quintiles["ESG_Risk_Score"], quintiles["YTD_Return_Pct"])
print(f"Correlation: {corr:.3f}, p-value: {p_value:.3f}")

Correlation: -0.003, p-value: 0.955


In [9]:
# Simple linear regression
import numpy as np
slope, intercept, r_value, p_value, std_err = stats.linregress(quintiles["ESG_Risk_Score"], quintiles["YTD_Return_Pct"])
print(f"Slope: {slope:.4f}, R²: {r_value**2:.4f}, p-value: {p_value:.4f}")

Slope: -0.0120, R²: 0.0000, p-value: 0.9548


In [10]:
# Sector-level ANOVA - do returns differ significantly by sector?
sector_groups = [group["YTD_Return_Pct"].values for name, group in quintiles.groupby("Sector")]
f_stat, p_val = stats.f_oneway(*sector_groups)
print(f"F-statistic: {f_stat:.2f}, p-value: {p_val:.4f}")

F-statistic: 6.80, p-value: 0.0000


In [11]:
# Sector-level ESG risk averages (this feeds Finding 2)
quintiles.groupby("Sector")["ESG_Risk_Score"].mean().sort_values(ascending=False).round(1)

,ESG_Risk_Score
Sector,
Energy,31.8
Basic Materials,26.7
Utilities,26.7
Consumer Defensive,25.4
Industrials,24.0
Financial Services,21.1
Healthcare,20.6
Communication Services,20.1
Consumer Cyclical,19.2


In [12]:
# Simulate excluding the worst ESG-risk quintile (Quintile 5) and see the effect on sector weights + return
before = quintiles["Sector"].value_counts(normalize=True).round(3) * 100
after = quintiles[quintiles["ESG_Risk_Quintile"] != 5]["Sector"].value_counts(normalize=True).round(3) * 100

comparison = pd.DataFrame({"Before_Exclusion_%": before, "After_Exclusion_%": after}).fillna(0)
comparison["Change_pp"] = (comparison["After_Exclusion_%"] - comparison["Before_Exclusion_%"]).round(2)
print(comparison.sort_values("Change_pp"))

print("\nAvg return before exclusion:", round(quintiles["YTD_Return_Pct"].mean(), 2))
print("Avg return after exclusion:", round(quintiles[quintiles["ESG_Risk_Quintile"] != 5]["YTD_Return_Pct"].mean(), 2))

                        Before_Exclusion_%  After_Exclusion_%  Change_pp
Sector                                                                  
Energy                                 4.1                1.5       -2.6
Industrials                           14.4               12.3       -2.1
Basic Materials                        4.6                2.7       -1.9
Consumer Defensive                     7.7                6.0       -1.7
Utilities                              6.7                5.4       -1.3
Communication Services                 3.1                3.6        0.5
Healthcare                            12.3               13.8        1.5
Consumer Cyclical                     12.3               13.8        1.5
Real Estate                            6.7                8.4        1.7
Financial Services                    14.2               16.2        2.0
Technology                            13.9               16.2        2.3

Avg return before exclusion: 13.49
Avg return afte